In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : gold_delivery_analysis
# PURPOSE    : Delivery performance analysis
# SOURCE     : silver.orders, silver.customers
# DESTINATION: fincompliance_catalog.gold.delivery_analysis
# METRICS:
#   - On time vs late delivery rate by state
#   - Average delivery delay days by state
#   - Delivery performance by year
#   - States with worst delivery performance
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    avg,
    round as spark_round,
    when,
    sum as spark_sum,
    current_timestamp
)

In [0]:
# ============================================================
# 
# READ FROM SILVER
# ============================================================

df_orders = spark.table(f"{SILVER_DB}.orders")
df_customers = spark.table(f"{SILVER_DB}.customers")

print("=" * 60)
print("SILVER TABLES LOADED")
print("=" * 60)
print(f"orders    : {df_orders.count():,} rows")
print(f"customers : {df_customers.count():,} rows")

print("\nDelivery status breakdown:")
df_orders \
    .groupBy("delivery_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("\nDelivery delay statistics:")
df_orders \
    .filter(col("delivery_status").isin(
        ["on_time", "late"]
    )) \
    .select("delivery_delay_days") \
    .summary("min", "max", "mean") \
    .show()

In [0]:
# ============================================================
# CALCULATE DELIVERY ANALYSIS BY STATE
# Join orders with customers to get state info
# Filter only delivered orders (on_time + late)
# Group by state and year
# ============================================================

df_delivery = (
    df_orders
    .filter(col("delivery_status").isin(
        ["on_time", "late"]
    ))
    .join(
        df_customers.select(
            "customer_id",
            "customer_state",
            "customer_state_name",
            "customer_region"
        ),
        on="customer_id",
        how="left"
    )
    .groupBy(
        "customer_state",
        "customer_state_name",
        "customer_region",
        "order_year"
    )
    .agg(
        count("order_id").alias("total_orders"),
        spark_sum(
            when(col("delivery_status") == "on_time", 1)
            .otherwise(0)
        ).alias("on_time_orders"),
        spark_sum(
            when(col("delivery_status") == "late", 1)
            .otherwise(0)
        ).alias("late_orders"),
        spark_round(avg("delivery_delay_days"), 2)
            .alias("avg_delay_days")
    )
    .withColumn(
        "on_time_rate_pct",
        spark_round(
            (col("on_time_orders") /
             col("total_orders")) * 100, 2
        )
    )
    .withColumn(
        "late_rate_pct",
        spark_round(
            (col("late_orders") /
             col("total_orders")) * 100, 2
        )
    )
    .orderBy(col("on_time_rate_pct").asc())
)

print("States with WORST delivery performance:")
df_delivery \
    .select(
        "customer_state_name",
        "order_year",
        "total_orders",
        "on_time_rate_pct",
        "late_rate_pct",
        "avg_delay_days"
    ) \
    .orderBy("on_time_rate_pct") \
    .show(15, truncate=False)

print(f"\nTotal rows : {df_delivery.count()}")

In [0]:
# ============================================================
# ADD GOLD AUDIT COLUMN
# ============================================================

df_delivery = df_delivery \
    .withColumn("gold_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_delivery.columns:
    print(f"  {col_name}")
print(f"Total rows : {df_delivery.count()}")

In [0]:
# ============================================================
# WRITE TO GOLD
# ============================================================

(
    df_delivery.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.delivery_analysis")
)

print(f"Written to {GOLD_DB}.delivery_analysis")
print(f"Rows : {df_delivery.count()}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("GOLD.DELIVERY_ANALYSIS VERIFICATION")
print("=" * 60)

df_gold = spark.table(f"{GOLD_DB}.delivery_analysis")
print(f"Total rows    : {df_gold.count()}")
print(f"Total columns : {len(df_gold.columns)}")

print("\nOverall delivery performance:")
df_gold \
    .agg(
        spark_round(avg("on_time_rate_pct"), 2)
            .alias("avg_on_time_rate"),
        spark_round(avg("late_rate_pct"), 2)
            .alias("avg_late_rate"),
        spark_round(avg("avg_delay_days"), 2)
            .alias("avg_delay_days")
    ) \
    .show(truncate=False)

print("\nBest performing states (2018):")
df_gold \
    .filter(col("order_year") == 2018) \
    .orderBy(col("on_time_rate_pct").desc()) \
    .select(
        "customer_state_name",
        "total_orders",
        "on_time_rate_pct",
        "late_rate_pct"
    ) \
    .show(5, truncate=False)

print("\nWorst performing states (2018):")
df_gold \
    .filter(col("order_year") == 2018) \
    .orderBy(col("on_time_rate_pct").asc()) \
    .select(
        "customer_state_name",
        "total_orders",
        "on_time_rate_pct",
        "late_rate_pct"
    ) \
    .show(5, truncate=False)

print("=" * 60)
print("gold.delivery_analysis verification complete!")
print("=" * 60)